# Kör — reproducible modelling

This notebook delegates to the reviewed scripts; it does not contain a separate training implementation. Only the delivered synthetic CSV and matching manifest are accepted. No customer logs, passwords, usernames, raw IPs or leaked data belong here.

All outputs are intentionally cleared. Scores describe this synthetic generator, not validated real-world detection or calibrated attack probabilities.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "src/models/train.py").is_file():
    raise RuntimeError("Open this notebook from the repository root.")
CSV = ROOT / "data/mvp_synthetic_logins.csv"
MANIFEST = ROOT / "data/mvp_synthetic_logins.manifest.json"
if not CSV.is_file() or not MANIFEST.is_file():
    raise RuntimeError("Wait for the delivered synthetic CSV and manifest; no placeholder training.")


## Train

Use the project `.venv` kernel with requirements installed. This cell runs the fixed 11-feature allowlist, account/episode/source/time-disjoint audit, fit-only preprocessing and validation-only threshold selection. Rows with missing or unsupported features abstain. Rule baseline, Logistic Regression, Random Forest and installed XGBoost are compared; LR is chosen for portable local deployment.


In [ ]:
subprocess.run([sys.executable, "-m", "src.models.train"], cwd=ROOT, check=True)


## Evaluate the held-out test split

No retraining occurs here. A changed CSV or manifest stops evaluation. Outputs include eligible support and abstentions, PR-AUC/AP, ROC-AUC, precision, recall, confusion matrix, FPR, scenario subgroups and the real 30/70 session policy. An infeasible validation threshold stays `null`.


In [ ]:
subprocess.run([sys.executable, "-m", "src.models.evaluate"], cwd=ROOT, check=True)


## Safe aggregate summary

Only aggregate metrics are displayed. Do not add `df.head()`, row previews, account/session identifiers or customer exports to notebook output. The generated holdout prediction file contains only synthetic event correlation IDs and predictions/status, not accounts, sessions, timestamps or labels.


In [ ]:
report = json.loads((ROOT / "outputs/evaluation.json").read_text())
summary = {name: {key: result[key] for key in ("eligible_rows", "abstentions", "pr_auc_average_precision", "roc_auc", "precision", "recall", "false_positive_rate")} for name, result in report["models"].items()}
print(json.dumps(summary, indent=2))


## Verify and read limitations

Run the meaningful regression suite and read `docs/model_results.md`. Browser parity inputs in `outputs/parity_cases.json` are feature-only selections from the actual synthetic holdout. The browser uses `models/demo_model.json`; `comparison_models.joblib` is only a locally created trusted training artifact, never an upload format. Session memory expiry is implemented separately by the demo. Synthetic source files remain on disk intentionally.


In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_models.py", "-v"], cwd=ROOT, check=True)
